## Open notebook in:
| Colab                               Gradient                                                                                                                                         |
|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolepcx/Transformers-in-Action/blob/main/CH05/ch_05_Train_Llama3_2_with_ORPO_and_LoRA.ipynb)                                              

# About this Notebook

This Jupyter notebook demonstrates the implementation of Odds Ratio Preference Optimization (ORPO) training for fine-tuning Llama-3.2-3B-Instruct. Below is an outline of some of the key concepts:

### **Tokenization and Dataset Preprocessing**
The dataset preparation involves:
- Configuring the tokenizer to support custom tokens (e.g., `<|finetune_right_pad_id|>`) and ensure compatibility with the model's architecture.
- Loading and splitting the `mlabonne/orpo-dpo-mix-40k` dataset into training and testing subsets, with only 20% of the training data utilized for this demonstration.
- Applying custom processing to structure prompts, chosen responses, and rejected responses using the Llama-3 chat template, ensuring the dataset adheres to ORPO requirements.

---

### **Model Configuration**
The notebook utilizes the `meta-llama/Llama-3.2-3B-Instruct` model with flash attention for memory-efficient training. Key configurations include:
- Enabling gradient checkpointing for reduced memory usage.
- Specifying a target module list for LoRA (Low-Rank Adaptation) fine-tuning, focusing on attention and projection layers critical for causal language modeling tasks.

---

### **Fine-Tuning with ORPO**
ORPO training aims to optimize reward preference for selected prompts. Key steps include:
- Defining `LoraConfig` to set LoRA parameters such as dropout, alpha, and rank (`r`).
- Setting training arguments using `ORPOConfig`, ensuring effective optimization with AdamW and scheduling strategies.
- Initializing the `ORPOTrainer`, which integrates model, dataset, and configurations for seamless training.


Refer to the [Llama prompt formats documentation](https://www.llama.com/docs/model-cards-and-prompt-formats/llama3_1) for additional details on chat template integration for Llama models.

# Install Dependencies

In [ ]:
%%capture
!pip install -qqq   transformers==4.47.1 \
                    bitsandbytes==0.45.0 \
                    peft==0.14.0 \
                    accelerate==1.2.1 \
                    datasets==3.2.0 \
                    trl==0.13.0 \
                    flash_attn==2.7.2.post1 \
                    wandb==0.19.1

In [ ]:
!pip install bitsandbytes==0.45.0 -qqq

# Set up

* lr and beta are the most important hyperparameters.
* we get the chat template from the tokenizer of the instruct version of SmolLM2

In [ ]:
import torch, os, multiprocessing
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed
)
from trl import ORPOTrainer, ORPOConfig


In [ ]:
set_seed(42)


In [ ]:
# Constants
COMPUTE_DTYPE = torch.bfloat16
ATTN_IMPLEMENTATION = "flash_attention_2"
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
PAD_TOKEN = "<|finetune_right_pad_id|>"
PAD_TOKEN_ID = 128004  # Based on the documentation

In [ ]:
bs = 4 #Batch size per device (training and validation)
gas = 8 #Gradient accumulation steps
mseqlen = 1024 #Maximum sequence length

lr = 1e-5 #Learning rate
beta = 0.1

lora_alpha = 32
lora_dropout = 0.0
lora_r = 16

output_dir = "./ORPO_LoRA_lr"+str(lr)+"_beta"+str(beta)+"/"

# Configure the tokenizer

In [ ]:
#Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = PAD_TOKEN
tokenizer.pad_token_id = PAD_TOKEN_ID
tokenizer.padding_side = "right"


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [ ]:
# Ensure the token is in the tokenizer's vocabulary
if tokenizer.pad_token not in tokenizer.vocab:
    tokenizer.add_special_tokens({"pad_token": tokenizer.pad_token})

# Preprocess the dataset

ORPOTrainer is like DPOTrainer. It expects a dataset with:
* a prompt
* a chosen answer to this prompt
* a rejected answer to this prompt


In [ ]:
# Load and split the dataset
ds = load_dataset("mlabonne/orpo-dpo-mix-40k", split="train").train_test_split(test_size=0.01)

# Use only 20% of the training set, adjust if you want to train on more data
ds_train = ds['train'].select(range(int(len(ds['train']) * 0.2)))
ds_test = ds['test']

# Print dataset sizes for confirmation
print(f"Train size: {len(ds_train)}, Test size: {len(ds_test)}")


README.md:   0%|          | 0.00/3.53k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/127M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/44245 [00:00<?, ? examples/s]

Train size: 8760, Test size: 443


In [ ]:
#Add the EOS token
def process(row):
    prompt_messages = tokenizer.apply_chat_template([row["chosen"][0]], tokenize=False)
    # Now we extract the final turn to define chosen/rejected responses
    chosen_messages = tokenizer.apply_chat_template(row["chosen"][1:], tokenize=False)+tokenizer.eos_token
    rejected_messages = tokenizer.apply_chat_template(row["rejected"][1:], tokenize=False)+tokenizer.eos_token
    row["prompt"] = prompt_messages
    row["chosen"] = chosen_messages
    row["rejected"] = rejected_messages
    return row

ds_train = ds_train.map(
    process,
    num_proc= multiprocessing.cpu_count(),
    load_from_cache_file=False,
)

ds_test = ds_test.map(
    process,
    num_proc= multiprocessing.cpu_count(),
    load_from_cache_file=False,
)

Map (num_proc=12):   0%|          | 0/8760 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/443 [00:00<?, ? examples/s]

# Load the model

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
      MODEL_NAME, device_map={"": 0}, torch_dtype=COMPUTE_DTYPE, attn_implementation=ATTN_IMPLEMENTATION)

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':True})

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

# Define the LoraConfig

In [ ]:
peft_config = LoraConfig(
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        r=lora_r,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules= ['k_proj', 'q_proj', 'v_proj', 'o_proj', "gate_proj", "down_proj", "up_proj"],
        modules_to_save=['embed_tokens',"lm_head"], # This is to train the embeddings of the special tokens of the chat template.
)

# Set the training arguments

*Note: Don't set the the max_prompt_length too high. ORPO training failed and diverged when I set it to 1024.*

In [ ]:
training_arguments = ORPOConfig(
        output_dir=output_dir,
        eval_strategy="steps",
        do_eval=True,
        optim="paged_adamw_8bit",
        per_device_train_batch_size=bs,
        gradient_accumulation_steps=gas,
        per_device_eval_batch_size=bs,
        log_level="debug",
        save_strategy="steps",
        save_steps=500,
        logging_steps=25,
        learning_rate=lr,
        bf16 = True,
        beta = beta,
        eval_steps=25,
        num_train_epochs=1,
        warmup_ratio=0.1,
        lr_scheduler_type="linear",
        max_length=mseqlen,
        max_prompt_length=512,
        dataset_num_proc=multiprocessing.cpu_count(),
        report_to="none",
)

In [ ]:
trainer = ORPOTrainer(
    model,
    peft_config=peft_config,
    args=training_arguments,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    processing_class=tokenizer,
)

/usr/local/lib/python3.10/dist-packages/trl/trainer/orpo_trainer.py:275: UserWarning: When using DPODataCollatorWithPadding, you should set `remove_unused_columns=False` in your TrainingArguments we have set it for you, but you should do it yourself in the future.
  warnings.warn(


Map (num_proc=12):   0%|          | 0/8760 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/8760 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/8760 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/443 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/443 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/443 [00:00<?, ? examples/s]

Using auto half precision backend


For preference optimization, the most important metrics in the training logs are Rewards/accuracies and Rewards/margins which should be increasing. For ORPO, since we are also learning to answer prompts, we want Logps/chosen to increase.

In [ ]:
trainer.train()

Currently training with a batch size of: 4
***** Running training *****
  Num examples = 8,760
  Num Epochs = 1
  Instantaneous batch size per device = 4
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 8
  Total optimization steps = 273
  Number of trainable parameters = 812,318,720


Step,Training Loss,Validation Loss,Runtime,Samples Per Second,Steps Per Second,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/rejected,Logps/chosen,Logits/rejected,Logits/chosen,Nll Loss,Log Odds Ratio,Log Odds Chosen
25,1.987100,1.911299,42.657700,10.385000,2.602000,-0.148191,-0.172605,0.626126,0.024414,-1.726049,-1.481906,-0.024981,0.064335,1.846898,-0.634921,0.305563
50,1.690200,1.579017,42.736500,10.366000,2.597000,-0.120968,-0.137780,0.605856,0.016812,-1.377800,-1.209678,-0.093632,-0.030933,1.513243,-0.647932,0.228812
75,1.433300,1.386199,42.729000,10.368000,2.598000,-0.100061,-0.111908,0.569069,0.011847,-1.119076,-1.000608,-0.226472,-0.182692,1.318769,-0.664157,0.175861
100,1.324400,1.297888,42.722600,10.369000,2.598000,-0.092154,-0.102663,0.553303,0.010509,-1.026632,-0.921542,-0.163741,-0.096579,1.229535,-0.673043,0.163186
125,1.265900,1.269355,42.732300,10.367000,2.598000,-0.090097,-0.100928,0.557808,0.010832,-1.009284,-0.900965,-0.065721,0.013604,1.201374,-0.669219,0.173142
150,1.209500,1.256464,42.729600,10.368000,2.598000,-0.088929,-0.100528,0.573574,0.011598,-1.005277,-0.889293,-0.021804,0.060573,1.189201,-0.661859,0.191143
175,1.205400,1.248207,42.736700,10.366000,2.597000,-0.088053,-0.100492,0.589339,0.012439,-1.004921,-0.880532,-0.022194,0.061511,1.181671,-0.654443,0.210779
200,1.221200,1.242483,42.741100,10.365000,2.597000,-0.087371,-0.100336,0.598348,0.012965,-1.003361,-0.873709,-0.005290,0.077745,1.176395,-0.649896,0.223164
225,1.254200,1.239043,42.737000,10.366000,2.597000,-0.087003,-0.100289,0.598348,0.013286,-1.002894,-0.870030,0.024790,0.110629,1.173221,-0.647213,0.230573
250,1.218000,1.237168,42.713800,10.371000,2.599000,-0.086754,-0.100186,0.598348,0.013432,-1.001861,-0.867538,0.037145,0.122661,1.171461,-0.646038,0.234273



***** Running Evaluation *****
  Num examples = 443
  Batch size = 4

***** Running Evaluation *****
  Num examples = 443
  Batch size = 4

***** Running Evaluation *****
  Num examples = 443
  Batch size = 4

***** Running Evaluation *****
  Num examples = 443
  Batch size = 4

***** Running Evaluation *****
  Num examples = 443
  Batch size = 4

***** Running Evaluation *****
  Num examples = 443
  Batch size = 4

***** Running Evaluation *****
  Num examples = 443
  Batch size = 4

***** Running Evaluation *****
  Num examples = 443
  Batch size = 4

***** Running Evaluation *****
  Num examples = 443
  Batch size = 4

***** Running Evaluation *****
  Num examples = 443
  Batch size = 4
Saving model checkpoint to ./ORPO_LoRA_lr1e-05_beta0.1/checkpoint-273
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Llama-3.2-3B-Instruct/snapshots/0cb88a4f764b7a12671c53f0838cd831a0843b95/config.json
Model config LlamaConfig {
  "architectures

TrainOutput(global_step=273, training_loss=1.3700413826184394, metrics={'train_runtime': 3187.502, 'train_samples_per_second': 2.748, 'train_steps_per_second': 0.086, 'total_flos': 0.0, 'train_loss': 1.3700413826184394, 'epoch': 0.9972602739726028})